# Week 3 — Wednesday: Choosing the Right Plot

**DATA 202 · Calvin University**

Same dataset as Monday — people experiencing homelessness — now cleaned and ready to visualize. A table of numbers tells you very little on its own; the right chart can reveal a pattern instantly, and the wrong chart can hide or distort it just as fast.

**Today's plan (~50 min of content — class is 65 min, ~15 min goes to the retrieval quiz and announcements):**

| Time | Section |
|---|---|
| ~5 min | Quick clean, reload, matching a question to a plot type |
| ~10 min | Histograms |
| ~10 min | Scatter plots |
| ~10 min | Line plots |
| ~10 min | Bar charts |
| ~5 min | When charts deceive |

Same two stop-and-check cues as Monday: **🎯 Predict First** (guess before we run the code) and **🙋 Quick Check** (a quick verbal question — no code).

---
## Quick Clean, Then Load

We'll redo Monday's cleaning in one cell so this notebook stands on its own.

In [ ]:
import pandas as pd
import plotly.express as px

homeless = pd.read_csv("../../datasets/homeless.csv")

homeless["city"] = (
    homeless["city"].str.strip().str.replace("-", " ", regex=False)
    .str.replace(r"[^a-zA-Z\s]", "", regex=True).str.title()
    .replace({"Sf": "San Francisco", "La": "Los Angeles"})
)
homeless["shelter_status"] = (
    homeless["shelter_status"].str.strip().str.lower()
    .str.replace(r"^sheltered$", "shelter", regex=True)
    .str.replace(r"shelter\s*,\s*pending", "shelter pending", regex=True)
    .str.replace(r"temporary shelter", "shelter temporary", regex=True)
)
homeless["education_level"] = (
    homeless["education_level"].str.strip().str.lower()
    .replace({"none": "None", "primary": "Primary", "secondary": "Secondary", "higher": "Higher"})
)

homeless.head()

---
## Matching a Question to a Plot Type (SLO 03C)

| Question | Variable types | Plot |
|---|---|---|
| What's the distribution of one number? | one numeric | **Histogram** |
| How do two numbers relate to each other? | two numeric | **Scatter** |
| How does something change across an order? | numeric, ordered (time, rank, duration...) | **Line** |
| How do groups compare? | numeric + categorical | **Bar** |

Same underlying idea every time: a column maps to a **visual channel** (x, y, color...), and the mapping you choose decides what question the chart can answer.

🙋 **Quick Check:** you want to compare the *average* `monthly_support_usd` across the five cities. Which plot type from the table fits, and why?

---
## Histograms · ~10 min

A **histogram** divides a numeric variable into **bins** and counts how many rows fall into each. It's the tool for seeing a variable's **center, spread, and shape** — symmetric? skewed? multiple peaks? — and for spotting outliers.

A histogram is *not* the same as a bar chart: a bar chart's bars are categories; a histogram's bars are *ranges of a number*.

🎯 **Predict First:** before we plot it — do you expect `monthly_support_usd` to be roughly symmetric, or skewed toward one side? Take a guess.

In [ ]:
px.histogram(homeless, x="monthly_support_usd", nbins=15,
             title="Distribution of Monthly Support ($)",
             labels={"monthly_support_usd": "Monthly Support (USD)"})

Try changing `nbins` to `5`, then to `40`. What do you gain, and what do you lose, at each? Whose interests might be served by a smoothed-out distribution instead of a spiky one — or the other way around?

---
### 🔨 Task 1 — Read a Histogram (~4 min)

Plot a histogram of `years_homeless`. Is it roughly symmetric, or skewed? What would that shape mean for a program planning shelter capacity?

In [ ]:
# Your code here


---
## Scatter Plots · ~10 min

A **scatter plot** shows the relationship between two numeric variables — one point per row. Useful for spotting trends, clusters, and outliers, and for asking whether one variable seems to predict another.

🎯 **Predict First:** do you expect `years_homeless` and `monthly_support_usd` to trend together (more time homeless → more support), trend apart, or show no clear relationship at all?

In [ ]:
px.scatter(homeless, x="years_homeless", y="monthly_support_usd", color="shelter_status",
           title="Years Homeless vs. Monthly Support",
           labels={"years_homeless": "Years Homeless", "monthly_support_usd": "Monthly Support (USD)"})

Was your prediction right? Does the pattern look different depending on `shelter_status`?

🙋 **Quick Check:** what does mapping `shelter_status` to `color=` add here that a plain black-and-white scatter plot couldn't show?

---
### 🔨 Task 2 — Build Your Own Scatter Plot (~4 min)

Plot `family_size` against `monthly_support_usd`, colored by `education_level`. Describe one pattern you see — or the lack of one.

In [ ]:
# Your code here


---
## Line Plots · ~10 min

Line plots connect points **in order** — almost always across time, but any meaningfully ordered variable works. We don't have dates here, so we'll order by `years_homeless` itself: as time homeless increases, how does *average* support change?

Notice we have to `groupby()` first — one line point per value of `years_homeless`, not one per person. Monday's skill feeds directly into today's chart.

🎯 **Predict First:** do you expect average support to rise, fall, or stay flat as years homeless increases?

In [ ]:
by_years = homeless.groupby("years_homeless")["monthly_support_usd"].mean().reset_index()

px.line(by_years, x="years_homeless", y="monthly_support_usd",
        title="Average Monthly Support by Years Homeless",
        labels={"years_homeless": "Years Homeless", "monthly_support_usd": "Average Monthly Support (USD)"})

🙋 **Quick Check:** this line jumps around a lot rather than following a smooth trend. What does that tell you about how much data we have *per* value of `years_homeless`? (Hint: think back to Monday's `groupby().count()`.)

---
### 🔨 Task 3 — Build Your Own Line Plot (~4 min)

Group by `years_homeless` again, but this time plot the average `family_size`. Does family size trend up, down, or stay flat as years homeless increases?

In [ ]:
# Your code here


---
## Bar Charts · ~10 min

**Bar charts** compare a number across categories. Vertical bars work well for a few categories; horizontal bars work better for many categories or long labels. `color=` turns one bar chart into a **grouped** or **stacked** comparison.

🎯 **Predict First:** which `education_level` group do you guess receives the highest *average* monthly support? Guess before running the cell.

In [ ]:
avg_support = homeless.groupby("education_level")["monthly_support_usd"].mean().reset_index()

px.bar(avg_support, x="education_level", y="monthly_support_usd",
       title="Average Monthly Support by Education Level",
       labels={"education_level": "Education Level", "monthly_support_usd": "Average Monthly Support (USD)"})

---
### 🔨 Task 4 — Build Your Own Bar Chart (~4 min)

Make a bar chart of the **count** of people per `city`, sorted from most to fewest. (*Hint:* `groupby("city").size()`, then `sort_values()`, then `reset_index()` before plotting — or pass `orientation="h"` if the city labels get cramped.)

In [ ]:
# Your code here


---
## When Charts Deceive · ~5 min

Charts can mislead in three broad ways:

* **Misrepresentation** — the chart is simply wrong: cherry-picked data, distorted proportions, a truncated axis that exaggerates a difference.
* **False impressions** — technically accurate, but a visual choice (3D effects, inconsistent colors, an unlabeled log scale) suggests a pattern that isn't really there.
* **Ambiguity** — missing labels, units, or context leave the chart open to multiple readings.

**Try it:** plot the *total* (not average) `monthly_support_usd` per `city`. Why might that number alone be misleading if you don't also show how many people are in each city?

In [ ]:
# Your code here — total monthly_support_usd per city


📎 **In class:** we'll pull up the **Graphics Principles cheat sheet** (the Novartis one) and go through it together — a one-page reference for exactly these misrepresentation/false-impression/ambiguity traps, and how to avoid them in your own charts. Keep it handy for the Final Project.

The **"Thumper Principle"** (borrowed from *Bambi*): if a chart can't say something useful, don't make it. A visualization exists to help people **see** the data more clearly — if it isn't doing that, it isn't finished yet.

---
## Coming Up

| Day | Topic | Builds on today |
|---|---|---|
| Fri | Forum 1 — *Counting*, Ch. 1 | Every chart today made a choice about what to show — this week's reading asks who makes that choice, and for whom |
| Week 4 | Joining tables | Combining datasets *before* you can plot them together |
| Week 5 | Clustering | Finding groups the data suggests, instead of ones we chose (like `city` or `education_level`) in advance |